## **Task 1: Custom Document Loader**

In [1]:
pip install langchain PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.9 MB/s eta 0:00:00


In [2]:
!pip install -U langchain langchain-text-splitters PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2

def load_and_chunk_document(file_path, chunk_size=300, overlap=50):
    # Step 1: Read the file's raw text
    if file_path.lower().endswith(".pdf"):
        text = ""
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() or ""
    else:  # assume .txt
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

    # Step 2: Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(text)

    # Step 3: Return the chunks
    return chunks

In [4]:
with open("sample.txt", "w") as f:
    f.write("This is a sample document. " * 50)

chunks = load_and_chunk_document("sample.txt", chunk_size=100, overlap=20)
print(f"Number of chunks: {len(chunks)}")
print("First chunk:", chunks[0])

Number of chunks: 17
First chunk: This is a sample document. This is a sample document. This is a sample document. This is a sample


## **Task 2: Custom Embedding Function**

In [5]:
!pip install -U sentence-transformers

In [6]:
from sentence_transformers import SentenceTransformer

def create_embeddings(chunks, model_name="all-MiniLM-L6-v2"):
    # Step 1: Load the model
    model = SentenceTransformer(model_name)

    # Step 2: Create embeddings for all chunks at once
    embeddings = model.encode(chunks)

    # Step 3: Return as list of lists (not numpy arrays)
    return embeddings.tolist()

In [7]:
embeddings = create_embeddings(chunks)
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding length (dimensions) per chunk: {len(embeddings[0])}")
print("First few numbers of first embedding:", embeddings[0][:5])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of embeddings: 17
Embedding length (dimensions) per chunk: 384
First few numbers of first embedding: [-0.05047730356454849, 0.08581642061471939, 0.008383268490433693, 0.0019401744939386845, 0.10290074348449707]


## **Task 3: Search Function**

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_chunks(query, chunks, embeddings, k=3):
    # Step 1: Embed the query using the same model
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = model.encode([query])

    # Step 2: Calculate cosine similarity between query and all chunk embeddings
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Step 3: Get indices of top-k highest similarity scores
    top_k_indices = np.argsort(similarities)[::-1][:k]

    # Step 4: Return the top-k chunks (with their scores, useful for the app later)
    results = [(chunks[i], similarities[i]) for i in top_k_indices]
    return results

In [9]:
results = search_chunks("What is this document about?", chunks, embeddings, k=3)
for i, (chunk, score) in enumerate(results):
    print(f"Result {i+1} (score: {score:.4f}): {chunk}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Result 1 (score: 0.6168): This is a sample document. This is a sample document.

Result 2 (score: 0.4894): This is a sample document. This is a sample document. This is a sample document. This is a sample

Result 3 (score: 0.4894): This is a sample document. This is a sample document. This is a sample document. This is a sample



## **Task 4: Streamlit App**


In [10]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 107.4 MB/s eta 0:00:00


In [11]:
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &>/content/streamlit_logs.txt &

In [13]:
import time
time.sleep(10)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [14]:
!cat /content/streamlit_logs.txt



2026-07-20 13:13:19.045 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.90.88.37:8501



## **Task 5: Add Cohere Answer Generation**

In [15]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.8 MB/s eta 0:00:00


In [16]:
!grep "model=" functions.py

        model="command-a-03-2025",
